# 04f — Modèle final de la Phase 4

Seul notebook à présenter : il reprend ce que les notebooks 04a à 04e ont établi et le compare à la Phase 3.

- **Features** (04e) : 31 features, dont l'écart à l'état initial du moteur (calculé sur ses 20 premiers cycles).
- **Hyperparamètres** (04d) : recherche Optuna refaite sur ces 31 features. Règle écrite avant tout résultat : on change les réglages si le gain de RMSE sur 10 découpages neufs est ≥ 0,3 cycle, avec t apparié et Wilcoxon p < 0,05 et un score NASA non dégradé. *Exception assumée, décidée après coup par l'équipe : le gain est sous 0,3 mais significatif sur 10 graines sur 10, on adopte quand même le candidat.*
- **Alerte** (04b) : alerte « panne dans les N prochains cycles » pour N = 15, 30 et 50, rappel minimal 0,99, seuil choisi sur la validation.
- **Test officiel** : lu une seule fois par modèle, à la fin.

In [ ]:
import sys
sys.path.insert(0, "..")  # pour importer src/ depuis notebooks/

import time
import numpy as np
import pandas as pd
import optuna
from scipy import stats
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, max_error, mean_absolute_error, median_absolute_error,
    precision_recall_curve, precision_score, r2_score, roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from xgboost import XGBRegressor

from src.preprocessing import (
    load_fd001, get_sensor_columns, find_constant_sensors,
    normalize_minmax, lisser_capteurs, calculer_rul, plafonner_rul, DATA_DIR,
)

FENETRE = 5         # fenetre de lissage (Phase 1)
PLAFOND_RUL = 125   # plafond de la RUL (Phase 2)
K_INIT = 20         # cycles servant a definir l'etat initial d'un moteur (04e)
GRAINE = 42
REGLAGES_PHASE3 = {"n_estimators": 200, "max_depth": 5, "learning_rate": 0.05}

# --- train : capteurs constants retires, min-max, lissage, RUL plafonnee (memes etapes que la Phase 3) ---
train = load_fd001("train")
capteurs_constants = find_constant_sensors(train, get_sensor_columns(train.columns))
train = train.drop(columns=capteurs_constants)
capteurs = get_sensor_columns(train.columns)  # les 15 capteurs utiles
train, scaler = normalize_minmax(train, capteurs)
train = lisser_capteurs(train, capteurs, fenetre=FENETRE)
train = calculer_rul(train)
train = plafonner_rul(train, plafond=PLAFOND_RUL)


def ecart_etat_initial(df, capteurs, k):
    """
    Pour chaque capteur : valeur lissee du cycle courant moins la moyenne des k premiers cycles du moteur.

    Parameters
    ----------
    df : pandas.DataFrame
        Tableau trie par moteur puis par cycle, capteurs deja lisses.
    capteurs : list de str
        Noms des capteurs.
    k : int
        Nombre de cycles de reference ; avant le cycle k, moyenne des cycles connus.

    Returns
    -------
    pandas.DataFrame
        Une colonne "<capteur>|ecartinit_<k>" par capteur, meme index que df.
    """
    unites = df["unit_number"].values
    colonnes = {}
    for capteur in capteurs:
        colonnes[f"{capteur}|ecartinit_{k}"] = np.zeros(len(df))  # espace reserve
    for moteur_id in np.unique(unites):  # un moteur a la fois : jamais de melange entre moteurs
        positions = np.where(unites == moteur_id)[0]
        for capteur in capteurs:
            valeurs = df[capteur].values[positions]
            somme_cumulee = np.cumsum(valeurs)
            moyenne_debut = somme_cumulee / np.arange(1, len(valeurs) + 1)  # cycle < k : moyenne des cycles connus
            if len(valeurs) >= k:
                moyenne_debut[k:] = somme_cumulee[k - 1] / k                # cycle >= k : moyenne figee des k premiers
            colonnes[f"{capteur}|ecartinit_{k}"][positions] = valeurs - moyenne_debut
    return pd.DataFrame(colonnes, index=df.index)


def features_31(df):
    """Jeu retenu au 04e : 15 capteurs lisses, time_cycles, 15 ecarts a l'etat initial."""
    return pd.concat([df[capteurs + ["time_cycles"]], ecart_etat_initial(df, capteurs, K_INIT)], axis=1)


def features_46(df):
    """Jeu de la Phase 3 : 15 capteurs lisses, time_cycles, 15 moyennes et 15 ecarts-types glissants (5 cycles)."""
    par_moteur = df.groupby("unit_number")[capteurs]
    moyennes = par_moteur.transform(lambda s: s.rolling(FENETRE, min_periods=1).mean()).add_suffix("_glissante_moy")
    ecarts_types = par_moteur.transform(lambda s: s.rolling(FENETRE, min_periods=1).std()).fillna(0.0).add_suffix("_glissante_std")
    return pd.concat([df[capteurs + ["time_cycles"]], moyennes, ecarts_types], axis=1)


X31 = features_31(train)
X46 = features_46(train)
assert X31.shape[1] == 31 and X46.shape[1] == 46
y = train["RUL_cappee"]
groups = train["unit_number"]
plis = list(GroupKFold(n_splits=5, shuffle=True, random_state=GRAINE).split(X31, y, groups=groups))  # 5 plis par moteur

# causalite : la feature d'un cycle ne change pas si on supprime les cycles suivants du moteur
complet = ecart_etat_initial(train, capteurs, K_INIT)
for cycle_coupe in [10, 25, 60, 100]:
    morceau = train[(train["unit_number"] == 1) & (train["time_cycles"] <= cycle_coupe)]
    derniere_ligne = morceau.index[-1]
    assert np.allclose(ecart_etat_initial(morceau, capteurs, K_INIT).loc[derniere_ligne], complet.loc[derniere_ligne]), \
        f"la feature du cycle {cycle_coupe} depend du futur"
print(f"{X46.shape[1]} features (Phase 3) et {X31.shape[1]} features (04e) ; causalite verifiee ; 5 plis par moteur.")

In [ ]:
# --- Hyperparametres : Optuna sur les 31 features, puis confirmation sur 10 decoupages par moteur neufs ---
def nasa_moyen(y_true, y_pred):
    """Score NASA moyen par prediction : surestimer (ecart > 0) coute plus cher que sous-estimer."""
    ecart = np.asarray(y_pred) - np.asarray(y_true)
    return float(np.mean(np.where(ecart < 0, np.exp(-ecart / 13) - 1, np.exp(ecart / 10) - 1)))


def evaluer_reglage(params, plis_a_utiliser):
    """
    RMSE et score NASA moyen par pli (31 features) pour un jeu d'hyperparametres XGBoost.

    Parameters
    ----------
    params : dict
        Hyperparametres de XGBRegressor.
    plis_a_utiliser : list de (idx_train, idx_val)
        Plis par moteur.

    Returns
    -------
    dict
        Listes par pli : "rmse" et "nasa_moyen".
    """
    rmse_plis = []
    nasa_plis = []
    for idx_train, idx_val in plis_a_utiliser:
        modele = XGBRegressor(random_state=GRAINE, **params)  # modele neuf a chaque pli
        modele.fit(X31.iloc[idx_train], y.iloc[idx_train])
        y_val = y.iloc[idx_val].values
        y_pred = modele.predict(X31.iloc[idx_val])
        rmse_plis.append(np.sqrt(np.mean((y_val - y_pred) ** 2)))
        nasa_plis.append(nasa_moyen(y_val, y_pred))
    return {"rmse": rmse_plis, "nasa_moyen": nasa_plis}


def objectif(essai):
    """Fonction objectif d'Optuna : propose des hyperparametres, renvoie le RMSE moyen de validation."""
    params = {
        "max_depth": essai.suggest_int("max_depth", 2, 8),
        "learning_rate": essai.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators": essai.suggest_int("n_estimators", 50, 600),
        "subsample": essai.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": essai.suggest_float("colsample_bytree", 0.3, 1.0),
        "min_child_weight": essai.suggest_float("min_child_weight", 1, 30, log=True),
        "reg_lambda": essai.suggest_float("reg_lambda", 0.1, 30, log=True),
    }
    return float(np.mean(evaluer_reglage(params, plis)["rmse"]))


ref = evaluer_reglage(REGLAGES_PHASE3, plis)
optuna.logging.set_verbosity(optuna.logging.WARNING)
debut = time.time()
etude = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=GRAINE))
etude.optimize(objectif, n_trials=60)  # 60 essais = 300 entrainements
REGLAGES_FINAUX = dict(etude.best_trial.params)
print(f"Optuna : 60 essais en {(time.time() - debut) / 60:.1f} min. RMSE plis de recherche : "
      f"{np.mean(ref['rmse']):.3f} (Phase 3) -> {etude.best_value:.3f} (candidat)")

rmse_ref, rmse_cand, nasa_ref, nasa_cand = [], [], [], []
for graine in range(1000, 1010):  # 10 graines jamais utilisees dans la recherche
    plis_neufs = list(GroupKFold(n_splits=5, shuffle=True, random_state=graine).split(X31, y, groups=groups))
    resultat_ref = evaluer_reglage(REGLAGES_PHASE3, plis_neufs)
    resultat_cand = evaluer_reglage(REGLAGES_FINAUX, plis_neufs)  # memes plis pour les deux : comparaison appariee
    rmse_ref.append(np.mean(resultat_ref["rmse"]))
    rmse_cand.append(np.mean(resultat_cand["rmse"]))
    nasa_ref.append(np.mean(resultat_ref["nasa_moyen"]))
    nasa_cand.append(np.mean(resultat_cand["nasa_moyen"]))
gains = np.array(rmse_ref) - np.array(rmse_cand)  # positif = le candidat est meilleur
p_t = stats.ttest_rel(rmse_ref, rmse_cand).pvalue
p_w = stats.wilcoxon(rmse_ref, rmse_cand).pvalue
print(f"Plis neufs : gain moyen {gains.mean():+.3f} cycle ({int(np.sum(gains > 0))}/10 graines), t apparie p={p_t:.4f}, "
      f"Wilcoxon p={p_w:.4f}, NASA moyen {np.mean(nasa_ref):.2f} -> {np.mean(nasa_cand):.2f}")
regle = gains.mean() >= 0.3 and p_t < 0.05 and p_w < 0.05 and np.mean(nasa_cand) <= 1.05 * np.mean(nasa_ref)
print(f"Regle ecrite : {'CHANGER' if regle else 'GARDER'} les reglages. Decision de l'equipe : on adopte le candidat.")
print({nom: round(valeur, 4) if isinstance(valeur, float) else valeur for nom, valeur in REGLAGES_FINAUX.items()})

In [ ]:
# --- Evaluation finale : 46 features (Phase 3) contre 31 features (final), puis alerte pour N = 15, 30 et 50 ---
RAPPEL_MIN = 0.99
HORIZONS = [15, 30, 50]


def choisir_seuil(y_vrai, score, rappel_min):
    """
    Parmi les seuils qui atteignent le rappel minimal, celui qui maximise la precision
    (a precision egale, le plus severe). Alerte si score >= seuil.

    Parameters
    ----------
    y_vrai : array-like de 0/1
        Etiquettes vraies de la validation (jamais du test).
    score : array-like
        Score continu : plus il est haut, plus le moteur est urgent.
    rappel_min : float
        Rappel minimal exige.

    Returns
    -------
    float
        Le seuil retenu.
    """
    precisions, rappels, seuils = precision_recall_curve(y_vrai, score)
    meilleur_seuil = None
    meilleure_precision = -1.0
    for i in range(len(seuils)):  # un candidat de seuil a la fois
        if rappels[i] >= rappel_min:
            if precisions[i] > meilleure_precision or (precisions[i] == meilleure_precision and seuils[i] > meilleur_seuil):
                meilleure_precision = precisions[i]
                meilleur_seuil = seuils[i]
    return meilleur_seuil


# --- test officiel : dernier cycle de chaque moteur, vraie RUL dans RUL_FD001.txt ---
test = load_fd001("test")
test = test.drop(columns=capteurs_constants)
test[capteurs] = scaler.transform(test[capteurs])  # transform seul : scaler appris sur le train
test = lisser_capteurs(test, capteurs, fenetre=FENETRE)
derniers = test.groupby("unit_number")["time_cycles"].idxmax().sort_index().values
y_test = pd.read_csv(f"{DATA_DIR}/RUL_FD001.txt", sep=r"\s+", header=None)[0].values.astype(float)
assert len(derniers) == 100 and len(y_test) == 100

modeles = {
    "46 features (Phase 3)": (X46, features_46(test).loc[derniers], REGLAGES_PHASE3),
    "31 features (final)": (X31, features_31(test).loc[derniers], REGLAGES_FINAUX),
}
regression = []
alertes = []


def metriques_regression(y_true, y_pred):
    """Metriques de regression du notebook 04a (celles gardees) : RMSE, MAE, R2, MedAE, MaxErr, biais, NASA moyen, surestimations > 20."""
    ecarts = y_pred - y_true  # positif = surestimation
    return {
        "RMSE": float(np.sqrt(np.mean(ecarts ** 2))), "MAE": mean_absolute_error(y_true, y_pred), "R2": r2_score(y_true, y_pred),
        "MedAE": median_absolute_error(y_true, y_pred), "MaxErr": max_error(y_true, y_pred), "Biais": float(np.mean(ecarts)),
        "NASA_moy": nasa_moyen(y_true, y_pred), "Sur20_%": float(100.0 * np.mean(ecarts > 20)),
    }


for nom, (feat_train, feat_test, params) in modeles.items():
    pred_val = np.zeros(len(y))  # predictions hors-pli sur la validation
    for idx_train, idx_val in plis:
        modele = XGBRegressor(random_state=GRAINE, **params)
        modele.fit(feat_train.iloc[idx_train], y.iloc[idx_train])
        pred_val[idx_val] = modele.predict(feat_train.iloc[idx_val])
    modele = XGBRegressor(random_state=GRAINE, **params).fit(feat_train, y)  # entraine sur les 100 moteurs du train
    pred_test = np.clip(modele.predict(feat_test), 0, None)  # RUL negative sans sens physique

    regression.append({"modele": nom, "jeu": "validation", **metriques_regression(y.values, pred_val)})
    regression.append({"modele": nom, "jeu": "test (100 moteurs)", **metriques_regression(y_test, pred_test)})

    for n in HORIZONS:
        seuil_cycles = -choisir_seuil((y.values <= n).astype(int), -pred_val, RAPPEL_MIN)  # alerte si RUL predite <= seuil
        etiquettes_val = (y.values <= n).astype(int)
        alerte_val = (pred_val <= seuil_cycles).astype(int)
        tn_v, fp_v, fn_v, tp_v = confusion_matrix(etiquettes_val, alerte_val, labels=[0, 1]).ravel()
        etiquettes_test = (y_test <= n).astype(int)
        alerte_test = (pred_test <= seuil_cycles).astype(int)
        tn, fp, fn, tp = confusion_matrix(etiquettes_test, alerte_test, labels=[0, 1]).ravel()
        alertes.append({
            "modele": nom, "N": n, "alerte si RUL predite <=": round(float(seuil_cycles), 1),
            "AUC (val)": roc_auc_score(etiquettes_val, -pred_val), "precision (val)": tp_v / (tp_v + fp_v), "fausses alertes (val)": int(fp_v),
            "en danger (test)": int(etiquettes_test.sum()), "detectes": int(tp), "rates": int(fn), "fausses alertes (test)": int(fp),
            "accuracy (test)": accuracy_score(etiquettes_test, alerte_test), "precision (test)": precision_score(etiquettes_test, alerte_test, zero_division=0),
            "F1 (test)": f1_score(etiquettes_test, alerte_test, zero_division=0),
        })

pd.set_option("display.width", 250)
print("RECAP 1 - erreur de prediction (metriques du notebook 04a)")
print(pd.DataFrame(regression).round(3).to_string(index=False))
print("\nScore NASA en somme sur le test (comparable aux articles) : "
      + ", ".join(f"{r['modele']} = {r['NASA_moy'] * 100:.1f}" for r in regression if r["jeu"].startswith("test")))
print("\nRECAP 2 - alerte (rappel minimal 0,99, seuil choisi sur la validation)")
print(pd.DataFrame(alertes).round(3).to_string(index=False))

In [ ]:
# --- Figures pour la presentation : modele final (31 features), test lu une seule fois (predictions de la cellule precedente) ---
import matplotlib.pyplot as plt

seuils = {n: -choisir_seuil((y.values <= n).astype(int), -pred_val, RAPPEL_MIN) for n in HORIZONS}  # seuils choisis sur la validation

fig, axes = plt.subplots(1, 4, figsize=(20, 4.6))
for ax, n in zip(axes[:3], HORIZONS):
    vrai = (y_test <= n).astype(int)
    alerte = (pred_test <= seuils[n]).astype(int)
    matrice = confusion_matrix(vrai, alerte, labels=[0, 1])
    ax.imshow(matrice, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, matrice[i, j], ha="center", va="center", fontsize=16, color="white" if matrice[i, j] > matrice.max() / 2 else "black")
    ax.set_xticks([0, 1], ["pas d'alerte", "alerte"])
    ax.set_yticks([0, 1], ["sain", "en danger"])
    ax.set_title(f"N = {n} (alerte si RUL predite <= {seuils[n]:.1f})")

ax = axes[3]
ax.scatter(y_test, pred_test, s=18, alpha=0.7)
ax.plot([0, 150], [0, 150], "k--", linewidth=1, label="prediction parfaite")
for n, couleur in zip(HORIZONS, ["tab:red", "tab:orange", "tab:green"]):
    ax.axhline(seuils[n], color=couleur, linewidth=1, linestyle=":", label=f"seuil N={n}")
ax.set_xlabel("RUL vraie (cycles)")
ax.set_ylabel("RUL predite (cycles)")
ax.set_title("Test : 100 moteurs")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
